In [16]:
import ibm_db
import ibm_db_dbi
import configparser
import pandas as pd

config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\DB2STAT_connect.ini')

conn_str=config['DB2STAT']['conn_str']
ibm_db_conn = ibm_db.connect(conn_str,'','')
print("SUCCESS")

In [2]:
# Fetch data using ibm_db_dbi

# PASTE DB2 QUERY HERE
# QUERY="""SELECT * FROM (
#  SELECT  distinct a.HOTEL_ID_VALUE, a.COMPANY_ID_VALUE, A.DATE_RANGE_FROM_DATE, A.DATE_RANGE_TO_DATE, B.CNCLLTN_POLICY_RELATIV_DEADLIN 
#   FROM SAL_RS.SAL_RS_NEGOTIATED_RATE A
# JOIN SAL_RS.SAL_RS_PRICING_PER_DATE_RANGE C ON C.OR_RATE_VALUE = A.ID_VALUE
# JOIN  SAL_RS.SAL_RS_RATE_CONDITIONS B   ON C.RATE_CONDITION_VALUE = b.ID_VALUE 
#  where

#   year(C.DATE_RANGE_FROM_DATE) = 2018 and B.CNCLLTN_POLICY_RELATIV_DEADLIN is not null ) a WHERE A.COMPANY_ID_VALUE = 29902
#   GROUP BY  a.HOTEL_ID_VALUE, a.COMPANY_ID_VALUE, A.DATE_RANGE_FROM_DATE, A.DATE_RANGE_TO_DATE, A.CNCLLTN_POLICY_RELATIV_DEADLIN"""

# conn = ibm_db_dbi.Connection(ibm_db_conn)
# db2_df = pd.read_sql(QUERY, conn)
# print(db2_df.head())

In [109]:
import pyexasol
import configparser

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\ExasolPROD.ini')

In [110]:
dsn=config['exasolPROD']['dsn']
user=config['exasolPROD']['user']
pwd=config['exasolPROD']['pwd']
schema=config['exasolPROD']['schema']
# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)

In [111]:
# PASTE EXASOL QUERY HERE
sql_query =  """
select hotel_id, 
f_key, 
booking_id,
BOOKING_CANCELLATION_DATE, sum(AMOUNT_TURNOVER) as TURNOVER,

case when to_timestamp(booking_cancellation_date) = to_timestamp(hotel_arrival_date) then
   (case when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) = 18 then '6pm DOA (00 hrs)' 
        when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) between 16 and 18 then '4pm DOA (02 hrs)' 
        when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) between 14 and 16 then '2pm DOA (04 hrs)' 
        when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) between 12 and 14 then '12pm DOA (06 hrs)' 
        when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) between 6 and 12 then '6am DOA (12 hrs)' 
        when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) between 0 and 6 then '6pm 1 day (24 hrs)' 
  end) 
when to_timestamp(booking_cancellation_date) = ADD_DAYS(to_timestamp(hotel_arrival_date), -1)  then
   (case when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) >=18  then '6pm 1 day (24 hrs)' 
         when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) between 12 and 18 then '12pm 1 day (30 hrs)' 
         when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) between 6 and 12 then '6am 1 day (36 hrs)' 
         when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) <=6 then '6am 2 days (48 hrs)' 
end)  

when to_timestamp(booking_cancellation_date) = ADD_DAYS(to_timestamp(hotel_arrival_date), -2)  then
   (case when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) >=18  then '6am 2 days (48 hrs)' 
        when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) <18 then '06 pm 3days (72 hrs)'
end ) 
 
when to_timestamp(booking_cancellation_date) = ADD_DAYS(to_timestamp(hotel_arrival_date), -3) then
   (case when hour(to_timestamp(booking_cancellation_date||' '||BOOKING_CANCELLATION_TIME, 'YYYY-MM-DD HH24:MI:SS')) >=18  then '06 pm 3days (72 hrs)' 
end) 
end as RANGE1
from DWHBIL.FAK_BOOKING   where YEAR(BOOKING_CANCELLATION_DATE ) = 2018  and BOOKING_STATUS_ID not in (0,1)
and    HOTEL_ID in (select distinct HOTEL_ID from DWHBIL.FAK_CRO_RATE where f_key in (select F_KEY from DWHBIL.V_REL_F_KEY_TO_MASTER_ACCOUNT where MASTER_ACCOUNT_ID ='00000000-0000-0000-0000-000000000000' ) and year(TRAVEL_DATE) = 2018)

and LOCAL.RANGE1 is not null
and to_date(BOOKING_CANCELLATION_DATE) between ADD_DAYS(to_date(HOTEL_ARRIVAL_DATE), -3)  and 
  HOTEL_ARRIVAL_DATE and F_KEY in (select F_KEY from DWHBIL.V_REL_F_KEY_TO_MASTER_ACCOUNT where MASTER_ACCOUNT_ID ='00000000-0000-0000-0000-000000000000' )
  group by booking_id, f_key,
  HOTEL_ID,
BOOKING_CANCELLATION_DATE, BOOKING_CANCELLATION_TIME, hotel_arrival_date"""
QUERY = connect.execute(sql_query)

# Importing data into a DataFrame
import pandas as pd
df_corsa = pd.DataFrame()
a=[]
for row in QUERY:
    a.append(row)
#print(len(a))
df_corsa = pd.DataFrame(a)
df_col_names = QUERY.col_names
df_corsa.columns = df_col_names
print(df_corsa.head())

In [112]:
df_corsa.count()

In [113]:
import pyexasol
import configparser

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\ExasolDET.ini')

dsn=config['exasolDET']['dsn']
user=config['exasolDET']['user']
pwd=config['exasolDET']['pwd']
schema=config['exasolDET']['schema']
# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)

In [114]:
# PASTE EXASOL QUERY HERE
sql_query =  """
SELECT
     CT.HOTEL_ID_VALUE
    ,CT.COMPANY_ID_VALUE

    ,TO_DATE(YEAR(TO_DATE(CT.DATE_RANGE_FROM_DATE, 'MM-DD-YYYY'))||'-'||MONTH(TO_DATE(CT.DATE_RANGE_FROM_DATE, 'MM-DD-YYYY'))||'-'||DAY(TO_DATE(CT.DATE_RANGE_FROM_DATE, 'MM-DD-YYYY')),'YYYY-MM-DD') AS DATE_RANGE_FROM_DATE
    ,TO_DATE(YEAR(TO_DATE(CT.DATE_RANGE_TO_DATE, 'MM-DD-YYYY'))||'-'||MONTH(TO_DATE(CT.DATE_RANGE_TO_DATE, 'MM-DD-YYYY'))||'-'||DAY(TO_DATE(CT.DATE_RANGE_TO_DATE, 'MM-DD-YYYY')),'YYYY-MM-DD') AS DATE_RANGE_TO_DATE
    ,CT.CNCLLTN_POLICY_RELATIV_DEADLIN AS CANCELLATION_POLICY_DB
    
    ,CT.CANCELLATION_TEXT AS CANCELLATION_TEXT
    ,DAYS_BETWEEN(local.DATE_RANGE_TO_DATE, local.DATE_RANGE_FROM_DATE) AS DAYS_DIFF
    ,(CT.DAYS_DB*24) + (18-ROUND(CT.HOURS_DIFFERENCE, 2)) AS DURATION_HOURS 
 FROM
(
 SELECT
       HOTEL_ID_VALUE, 
       COMPANY_ID_VALUE,
       DATE_RANGE_FROM_DATE,
       DATE_RANGE_TO_DATE,
       CNCLLTN_POLICY_RELATIV_DEADLIN, 
       COALESCE(CAST (REPLACE(REGEXP_SUBSTR(CNCLLTN_POLICY_RELATIV_DEADLIN, '(?<=P)(.*)(?=D)'), '-','') AS DECIMAL(10,0)), 0)  AS DAYS_DB,
       COALESCE(CAST (REPLACE(REGEXP_SUBSTR(CNCLLTN_POLICY_RELATIV_DEADLIN, '(?<=T)(.*)(?=H)'), '-','') AS DECIMAL(10,0)), 0)  AS HOURS_DB,
       COALESCE(CAST (REPLACE(REGEXP_SUBSTR(CNCLLTN_POLICY_RELATIV_DEADLIN, '(?<=H)(.*)(?=M)'), '-','') AS DECIMAL(10,0)), 0)  AS MINUTES_DB,
       COALESCE(CAST(REPLACE(REGEXP_SUBSTR(CNCLLTN_POLICY_RELATIV_DEADLIN, '(?<=M)(.*)(?=S)'), '-','') AS DECIMAL(10,0)), 0)  AS SECONDS,
       SECONDS_BETWEEN('2019-01-01 18:00:00',  ('2019-01-01 ' ||  LOCAL.HOURS_DB  || ':' ||  LOCAL.MINUTES_DB || ':' ||   LOCAL.SECONDS )) AS SECONDS_DIFFERENCE, 
       HOURS_BETWEEN('2019-01-01 18:00:00',  ('2019-01-01 ' ||  LOCAL.HOURS_DB  || ':' ||  LOCAL.MINUTES_DB || ':' ||   LOCAL.SECONDS ))  AS HOURS_DIFFERENCE,
       
       CASE 
            WHEN LENGTH(FLOOR((LOCAL.HOURS_DIFFERENCE - FLOOR(LOCAL.HOURS_DIFFERENCE)) * 60 )) =1 THEN '0' || FLOOR((LOCAL.HOURS_DIFFERENCE - FLOOR(LOCAL.HOURS_DIFFERENCE)) * 60 ) 
            ELSE FLOOR((LOCAL.HOURS_DIFFERENCE - FLOOR(LOCAL.HOURS_DIFFERENCE)) * 60 )   
       END  AS MINUTES_EXA,
       CASE 
            WHEN LENGTH(ROUND(MOD( CAST(LOCAL.SECONDS_DIFFERENCE  AS numeric), 60), 3)) =1 THEN '0' || ROUND(MOD( CAST(LOCAL.SECONDS_DIFFERENCE  AS numeric), 60), 3) 
            ELSE ROUND(MOD( CAST(LOCAL.SECONDS_DIFFERENCE  AS numeric), 60), 3)   
       END  AS SECONDS_EXA,
       
       CASE 
            WHEN CAST(FLOOR(LOCAL.HOURS_DIFFERENCE)  AS numeric) > 12 THEN  'Until ' || LOCAL.DAYS_DB || ' days, ' ||  (  CAST(FLOOR(LOCAL.HOURS_DIFFERENCE)  AS numeric) -12)  ||':'|| LOCAL.MINUTES_EXA || ':' || LOCAL.SECONDS_EXA || ' PM possible' 
            WHEN  CAST(FLOOR(LOCAL.HOURS_DIFFERENCE)  AS numeric) =  12 THEN  'Until ' || LOCAL.DAYS_DB || ' days, ' ||  (  CAST(FLOOR(LOCAL.HOURS_DIFFERENCE)  AS numeric))  ||':'|| LOCAL.MINUTES_EXA|| ':' || LOCAL.SECONDS_EXA || ' PM possible' 
            ELSE 'Until ' || LOCAL.DAYS_DB || ' days, ' ||  FLOOR(LOCAL.HOURS_DIFFERENCE)  ||':'|| LOCAL.MINUTES_EXA || ':' || LOCAL.SECONDS_EXA || ' AM possible'  
       END  AS CANCELLATION_TEXT
       
 FROM TEMP.CXL_SIEMENS)CT"""
QUERY = connect.execute(sql_query)

# Importing data into a DataFrame
import pandas as pd
df = pd.DataFrame()
a=[]
for row in QUERY:
    a.append(row)
#print(len(a))
df_cxl = pd.DataFrame(a)
df_col_names = QUERY.col_names
df_cxl.columns = df_col_names
df_cxl = df_cxl.replace(to_replace=[r"\\t|\\n|\\r", "\t|\n|\r"], value=["",""], regex=True)
print(df_cxl.head())

In [115]:
df_cxl.count()

In [116]:
df_corsa.F_KEY.unique()

In [117]:
df_cxl.COMPANY_ID_VALUE.unique()

In [158]:
# MERGE

CXL_SIEMENS_df = pd.merge(left=df_corsa, 
                          right=df_cxl, 
                          #on='HOTEL_ID', 
                           right_on = ['HOTEL_ID_VALUE', 'COMPANY_ID_VALUE'],
                           left_on = ['HOTEL_ID', 'F_KEY'],
                          how='left')

CXL_SIEMENS_df

In [159]:
# Imputation
import numpy as np
import pandas as pd
CXL_SIEMENS_df['IMPUTED'] = CXL_SIEMENS_df.apply(lambda row:"Y" if pd.isnull(row['COMPANY_ID_VALUE']) else "N",
                                            axis=1)
CXL_SIEMENS_df['HOTEL_ID_VALUE'] = CXL_SIEMENS_df.apply(lambda row:row['HOTEL_ID'] if np.isnan(row['HOTEL_ID_VALUE']) else row['HOTEL_ID_VALUE'],
                                            axis=1)
CXL_SIEMENS_df['COMPANY_ID_VALUE'] = CXL_SIEMENS_df.apply(lambda row:row['F_KEY'] if np.isnan(row['COMPANY_ID_VALUE']) else row['COMPANY_ID_VALUE'],
                                            axis=1)
CXL_SIEMENS_df['DATE_RANGE_FROM_DATE'] = CXL_SIEMENS_df.apply(lambda row: '2018-01-01' if pd.isnull(row['DATE_RANGE_FROM_DATE']) else row['DATE_RANGE_FROM_DATE'],
                                            axis=1)
CXL_SIEMENS_df['DATE_RANGE_TO_DATE'] = CXL_SIEMENS_df.apply(lambda row: '2018-12-31' if pd.isnull(row['DATE_RANGE_TO_DATE']) else row['DATE_RANGE_TO_DATE'],
                                            axis=1)
CXL_SIEMENS_df['CANCELLATION_POLICY_DB'] = CXL_SIEMENS_df.apply(lambda row:'PT0S' if pd.isnull(row['CANCELLATION_POLICY_DB']) else row['CANCELLATION_POLICY_DB'],
                                            axis=1)
CXL_SIEMENS_df['CANCELLATION_TEXT'] = CXL_SIEMENS_df.apply(lambda row:'Until 0 days, 6:00:00 PM possible' if pd.isnull(row['CANCELLATION_TEXT']) else row['CANCELLATION_TEXT'],
                                            axis=1)
CXL_SIEMENS_df['DAYS_DIFF'] = CXL_SIEMENS_df.apply(lambda row: 364 if np.isnan(row['DAYS_DIFF']) else row['DAYS_DIFF'],
                                            axis=1)
CXL_SIEMENS_df['DURATION_HOURS'] = CXL_SIEMENS_df.apply(lambda row:0 if np.isnan(row['DURATION_HOURS']) else row['DURATION_HOURS'],
                                            axis=1)
CXL_SIEMENS_df[CXL_SIEMENS_df['IMPUTED'] == 'Y'].head()

In [210]:
import re
CXL_SIEMENS_bet['DURATION_HOURS'] = pd.to_numeric(CXL_SIEMENS_bet['DURATION_HOURS'] )
CXL_SIEMENS_bet['CXL_CLUSTER'] = pd.to_numeric(CXL_SIEMENS_bet['RANGE1'].str.split('(').str[1].str.split(' hrs').str[0])

CXL_SIEMENS_bet['CXL_ANALYSIS'] = CXL_SIEMENS_bet.apply(lambda row:'REFUND' if (row['CXL_CLUSTER']>=row['DURATION_HOURS']) else 'NON_REFUND',
                                            axis=1)


In [161]:
CXL_SIEMENS_bet = CXL_SIEMENS_df[CXL_SIEMENS_df['BOOKING_CANCELLATION_DATE'].between(CXL_SIEMENS_df['DATE_RANGE_FROM_DATE'], 
                                                                   CXL_SIEMENS_df['DATE_RANGE_TO_DATE'], 
                                                                   inclusive=False)]

In [213]:
CXL_SIEMENS_bet[CXL_SIEMENS_bet.groupby(['HOTEL_ID', 'F_KEY', 'BOOKING_CANCELLATION_DATE', 'BOOKING_ID'])['DAYS_DIFF'].rank() == 1].to_excel('C:\\Users\\USER\\Documents\\misc\\CXL_SIEMENS_ranked_fianl.xlsx', 
            sheet_name='CXL_analysis', 
            header=True, 
            encoding='utf-8', 
            index=False, 
            freeze_panes=(1,1))

In [211]:
CXL_SIEMENS_bet.groupby(['CXL_ANALYSIS']).count()